### Choose where this notebook runs

This is the **IQM edition**: the Estimator cell stays an exact local reference computation (IQM has
no Estimator primitive) and the same observable is then measured from counts — the Bell circuit is
transpiled to the IQM native gate set **PRX + CZ** and run on **Garnet** (20 qubits) or **Emerald**
(54 qubits) via IQM Resonance. Pick the backend
by editing the single `TARGET` line below (see `README.md` and `iqm_backend.py` in this folder):

| `TARGET` | runs on | token? | QPU credits? |
|---|---|---|---|
| `"fake_apollo"` | local 20-qubit noisy simulator | no | no |
| `"garnet:mock"` / `"emerald:mock"` | IQM Resonance cloud mock (pipeline check: returns RANDOM bitstrings, no physics) | yes | no |
| `"garnet"` / `"emerald"` | **real QPU** (20q / 54q) | yes | yes |


# Quantum Operators and Observables

**Course**: Quantum Computing Laboratory  
**Version**: 2.2 (Qiskit 2.2 Compatible)  
**Last Updated**: November 2025

---

In [ ]:
from iqm_backend import get_backend, run_counts   # helper module in this folder (see iqm_backend.py)
# run_counts(circuit, TARGET, shots=...) = transpile to the IQM native gates (PRX + CZ) + run + get_counts()
# get_backend(TARGET) returns the raw Qiskit backend object if you want to call backend.run yourself
TARGET = "fake_apollo"   # change this one line to choose the backend (see the table above)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `from iqm_backend import get_backend, run_counts` | the helper module of this folder: `run_counts(circuit, TARGET, shots=...)` = `transpile_to_IQM` (rewrite the circuit with the native `r` = PRX rotations and `cz` gates, routed on the chip's square lattice) → `backend.run` → counts as a plain dict; `run_counts(..., return_transpiled=True)` also returns the transpiled circuit |
| `TARGET = "fake_apollo"` | a local noisy simulator with the 20-qubit Garnet topology: no token, no credits, realistic gate and readout noise; `"garnet:mock"` checks the cloud pipeline but returns RANDOM bitstrings; `"garnet"` / `"emerald"` are the real QPUs (token + credits; IQM Resonance declines circuits above 10 000 native instructions) |
| `get_backend(TARGET)` | the raw Qiskit backend object (name, qubit count, coupling map) if you want to call `backend.run` yourself |

</details>


## 📚 Overview

Understand quantum operators, observables, and expectation values using Qiskit 2.2's modern **Estimator** primitive.

### Topics:
- Operator class and representations
- Pauli operators (X, Y, Z)
- Observable measurements
- Estimator primitive for expectation values
- Operator algebra

## 🎯 Learning Objectives

1. ✅ Understand quantum operators
2. ✅ Work with Pauli matrices
3. ✅ Measure observables
4. ✅ Calculate expectation values
5. ✅ Use EstimatorV2 primitive

In [ ]:
# Qiskit 2.2 imports
from qiskit import QuantumCircuit                                   # the object we build circuits with
from qiskit.quantum_info import Operator, Pauli, SparsePauliOp, Statevector   # exact linear algebra: unitary matrices, Pauli strings, weighted sums of Paulis, state vectors
from qiskit_aer import AerSimulator                                 # local classical simulator of a quantum computer
from qiskit_aer.primitives import EstimatorV2                       # Aer's Estimator primitive: expectation values <O> of a circuit (exact by default)
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager   # transpiler: circuit -> the gates/couplings a backend understands
import numpy as np                                                  # arrays, np.allclose, np.round
from IPython.display import display                      # show a figure (e.g. a circuit drawing) from the middle of a cell
def draw(qc, fold=-1, **kw): return qc.draw("mpl", style="clifford", fold=fold, **kw)   # every circuit picture: matplotlib drawer; fold=-1 = one long row, fold=N wraps after N columns

print('✓ Imports successful!')
print('  Qiskit 2.2 operators ready')


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qiskit.quantum_info` | the exact-linear-algebra toolbox: `Operator` (a matrix), `Pauli` (one Pauli string such as `'XZ'`), `SparsePauliOp` (a weighted sum of Pauli strings = an observable or a Hamiltonian), `Statevector` (the amplitudes of a circuit's output state) |
| `EstimatorV2` vs a Sampler | the two Qiskit primitives: a *Sampler* returns bitstrings, an *Estimator* returns the number ⟨ψ\|O\|ψ⟩ — on hardware it is built from many sampled bitstrings (we do this by hand later in the notebook) |
| `draw(qc)` | our one-line helper around `qc.draw("mpl", style="clifford")`; `display(...)` shows the figure even when the call is not the last line of the cell |

</details>


## Part 1: Operator Basics

In [ ]:
# Create operators from circuits
qc = QuantumCircuit(2)        # 2 qubits, no classical bits: we want the unitary, and a measurement is not a unitary
qc.h(0)                       # Hadamard on qubit 0: |0> -> (|0> + |1>)/sqrt2
qc.cx(0, 1)                   # CNOT, control qubit 0, target qubit 1: with the H it turns |00> into the Bell state
display(draw(qc)); print("↑ the Bell-state circuit whose 4×4 unitary we compute below")

# Convert to operator
U = Operator(qc)              # multiplies the gate matrices in circuit order: the unitary of the whole circuit

print('Circuit operator:')
print(f'  Shape: {U.dim}')                 # (input dimension, output dimension) = (4, 4) for 2 qubits
print(f'  Is unitary: {U.is_unitary()}')   # checks U†U = 1 numerically
print('\nMatrix representation:')
print(U.data)                              # the NumPy array; column k = U applied to basis state |k>, k = 00, 01, 10, 11 (qubit 0 is the RIGHTMOST bit)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `Operator(qc)` | turns a circuit into its matrix by multiplying the gate matrices (each embedded in the 2-qubit space) in execution order; `U.data` is the plain NumPy array, `U.dim` the pair of dimensions |
| `U.is_unitary()` | every circuit without measurements is a unitary: columns are orthonormal, U†U = 1 — the property that makes quantum evolution reversible |
| column order `00, 01, 10, 11` | Qiskit is *little-endian*: the rightmost bit is qubit 0. That is why the H (on qubit 0) shows up as the 2×2 blocks ±1/√2 inside the 4×4 matrix and the CNOT swaps the rows `01` and `11` |
| first column `[0.707, 0, 0, 0.707]` | U\|00⟩ = (\|00⟩ + \|11⟩)/√2: the Bell state Φ⁺ — reading the matrix column by column is reading what the circuit does to each basis state |

</details>


<details open><summary><b>📐 Working it out</b></summary>

Qiskit orders the two qubits as $|q_1 q_0\rangle$, so a gate on qubit 0 sits in the **right** factor of the tensor product:

$$ I\otimes H=\frac{1}{\sqrt2}\begin{pmatrix}1&1&0&0\\1&-1&0&0\\0&0&1&1\\0&0&1&-1\end{pmatrix},\qquad
\mathrm{CNOT}_{0\to1}=\begin{pmatrix}1&0&0&0\\0&0&0&1\\0&0&1&0\\0&1&0&0\end{pmatrix}
\ \ (\text{swaps }|01\rangle\leftrightarrow|11\rangle). $$

The circuit applies H first, then CNOT, so $U=\mathrm{CNOT}\,(I\otimes H)$: take the rows of $I\otimes H$ and swap rows 1 and 3:

$$ U=\frac{1}{\sqrt2}\begin{pmatrix}1&1&0&0\\0&0&1&-1\\0&0&1&1\\1&-1&0&0\end{pmatrix}, $$

exactly the matrix the cell prints ($1/\sqrt2 = 0.7071$). Its first column is $U|00\rangle=(|00\rangle+|11\rangle)/\sqrt2$, the Bell state.

</details>


## Part 2: Pauli Operators

In [ ]:
# Pauli matrices
X = Pauli('X')                # a Pauli object from its label; multi-qubit labels like 'XZ' work the same way
Y = Pauli('Y')
Z = Pauli('Z')

print('Pauli X:')
print(X.to_matrix())          # the 2x2 NumPy matrix: X = [[0,1],[1,0]] is the bit flip (quantum NOT)
print('\nPauli Z:')
print(Z.to_matrix())          # Z = diag(1, -1): a phase flip, eigenvalue +1 on |0> and -1 on |1>


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `Pauli('X')` | a symbolic Pauli operator (label + phase), cheaper than a matrix: it knows how to multiply, commute and tensor with other Paulis; `.to_matrix()` gives the explicit array when you need it |
| `X` = `[[0,1],[1,0]]` | the bit flip: X\|0⟩ = \|1⟩, X\|1⟩ = \|0⟩ (the NOT gate); its eigenstates are \|±⟩ = (\|0⟩ ± \|1⟩)/√2 with eigenvalues ±1 |
| `Z` = `diag(1, −1)` | the phase flip; measuring Z means reading the bit b and assigning the eigenvalue (−1)ᵇ — the rule we use later to turn counts into expectation values |

</details>


## Part 3: Observable Measurements

In [ ]:
# Create observable
observable = SparsePauliOp.from_list([('ZZ', 1.0), ('XX', 1.0)])   # O = 1.0*Z⊗Z + 1.0*X⊗X as a list of (Pauli label, coefficient)

print(f'Observable: {observable}')            # prints the labels and the (complex) coefficients
print(f'Number of terms: {observable.size}')  # 2 Pauli strings


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `SparsePauliOp.from_list([(label, coeff), ...])` | an observable as a weighted sum of Pauli strings — the natural form for any Hamiltonian; `observable.paulis` and `observable.coeffs` give the two columns back |
| `'ZZ'`, `'XX'` | two-qubit Pauli strings; the rightmost letter acts on qubit 0 (little-endian, like the matrix columns above). Z⊗Z asks "are the two bits equal?", X⊗X asks the same question in the \|±⟩ basis |
| `.size` | the number of terms: on hardware each distinct measurement basis costs one set of shots, so the size of an observable is a cost |

</details>


## Part 4: Expectation Values with Estimator

In [ ]:
# Prepare state
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)  # Bell state  |Φ+> = (|00> + |11>)/sqrt2

# Setup estimator
simulator = AerSimulator()                      # the backend the circuit is transpiled for
estimator = EstimatorV2()                       # Aer Estimator: computes <O> from the simulated state (exact: no shot noise by default)
pm = generate_preset_pass_manager(optimization_level=1, backend=simulator)   # transpiler pass manager for that backend
transpiled = pm.run(qc)                         # an "ISA" circuit: only gates the backend knows (Aer knows h and cx, so nothing changes)

# Calculate expectation
job = estimator.run([(transpiled, observable)])   # a list of "pubs" = (circuit, observable[, parameter values]) — here one pub
result = job.result()
expectation = result[0].data.evs                # .evs = expectation value(s) of pub 0; one observable -> one number

print(f'Expectation value: {expectation}')
print(f'Exact check with Statevector: {Statevector(qc).expectation_value(observable).real:.6f}')   # <ψ|O|ψ> computed directly from the amplitudes


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `estimator.run([(transpiled, observable)])` | the Estimator primitive takes a list of *pubs* (circuit, observable) and returns ⟨ψ\|O\|ψ⟩ for each; `result[0].data.evs` is the value of the first pub (an array when several observables or parameter sets are given) |
| `EstimatorV2()` from `qiskit_aer` | the local, exact version of the primitive (it contracts the simulated state with the observable — no sampling). On IBM hardware the same code uses `qiskit_ibm_runtime.EstimatorV2(backend)`, which estimates ⟨O⟩ from shots |
| `generate_preset_pass_manager(...)` / `pm.run(qc)` | transpilation to the backend's gate set: mandatory on hardware, a no-op on Aer here; kept so that the cell is the same in both worlds |
| `Statevector(qc).expectation_value(observable)` | the textbook formula ⟨ψ\|O\|ψ⟩ with the exact amplitudes — the reference against which every estimate from counts (next cell) is judged; `.real` drops the +0j |

</details>


<details open><summary><b>📐 Working it out</b></summary>

For the Bell state $|\Phi^+\rangle=(|00\rangle+|11\rangle)/\sqrt2$:

* $Z\otimes Z$ gives $(+1)(+1)=+1$ on $|00\rangle$ and $(-1)(-1)=+1$ on $|11\rangle$, so $Z\otimes Z|\Phi^+\rangle=|\Phi^+\rangle$ and $\langle ZZ\rangle = 1$.
* $X\otimes X$ flips both bits: $|00\rangle\leftrightarrow|11\rangle$, which leaves $|\Phi^+\rangle$ unchanged, so $\langle XX\rangle = 1$.

$$\langle O\rangle=\langle ZZ\rangle+\langle XX\rangle = 1+1 = 2.0 ,$$

the number printed by the Estimator and by the `Statevector` check. (The maximum possible value: $|\Phi^+\rangle$ is a common +1 eigenstate of both terms. For any product state $\langle ZZ\rangle+\langle XX\rangle=z_1z_2+x_1x_2\le1$ by Cauchy–Schwarz, so a value above 1 certifies entanglement: this observable is an *entanglement witness*.)

</details>


### The same expectation value from measurement counts

A quantum computer never returns ⟨O⟩ directly: it returns **bitstrings**. An Estimator on hardware is
built from Samplers. For each Pauli string of the observable we rotate every qubit into the eigenbasis of
its Pauli letter (H for X, S†H for Y, nothing for Z or I), measure in the computational basis, and read the
eigenvalue of the string as the **parity** of the measured bits: $(-1)^{\text{number of 1s}}$ on the
non-identity qubits. Averaging over shots gives ⟨P⟩; the coefficients combine the terms.


In [ ]:
# There is no Estimator primitive on IQM: expectation values are built from counts, exactly as on any real hardware.
def run_circuit(circuit, shots=4096):
    """Transpile + run on TARGET + counts in one call (also used by the In-depth analysis)."""
    return run_counts(circuit, TARGET, shots=shots)          # transpile_to_IQM (native PRX + CZ) -> backend.run -> dict {'bitstring': count}

def pauli_expectation(circuit, label, shots=4096):
    """<P> of the state prepared by `circuit` for one Pauli string `label` (e.g. 'XZ'), estimated from measurement counts."""
    meas = circuit.copy()                                  # never modify the state-preparation circuit itself
    for q, p in enumerate(reversed(label)):                # labels are little-endian: the LAST character is qubit 0, hence reversed
        if p == 'X':   meas.h(q)                           # X eigenbasis -> computational basis: H|+> = |0>, H|-> = |1>
        elif p == 'Y': meas.sdg(q); meas.h(q)              # Y eigenbasis -> computational basis: S† then H maps |+i>, |-i> to |0>, |1>
        # 'Z' and 'I': measure in the computational basis as it is
    meas.measure_all()                                     # adds a classical register and measures every qubit
    counts = run_circuit(meas, shots)
    total = sum(counts.values())
    expval = 0.0
    for bits, n in counts.items():                         # bits = 'q1q0' (little-endian, the same order as the label)
        parity = sum(int(b) for b, p in zip(bits, label) if p != 'I') % 2   # number of 1s on the qubits that carry X, Y or Z
        expval += (-1) ** parity * n / total               # eigenvalue +1 for even parity, -1 for odd, weighted by the frequency
    return expval

total = 0.0
for pauli, coeff in zip(observable.paulis, observable.coeffs):   # a SparsePauliOp is a list of (Pauli string, coefficient)
    ev = pauli_expectation(qc, pauli.to_label())
    total += coeff.real * ev
    print(f'<{pauli.to_label()}> from counts on {TARGET} = {ev:+.4f}   (coefficient {coeff.real:+.1f})')
print(f'Expectation value from counts: {total:.4f}   (exact Estimator above: {float(expectation):.4f})')


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `run_counts(circuit, TARGET, shots=...)` | the IQM route: `transpile_to_IQM` rewrites the Bell circuit plus basis changes into the native `r` (PRX) and `cz` gates on the chip's square lattice, `backend.run` executes it, and a plain dict of bitstrings comes back — wrapped in `run_circuit` for reuse |
| `meas.h(q)` / `meas.sdg(q); meas.h(q)` | basis changes before the measurement: a measurement can only read Z, so X and Y are rotated onto Z (H maps the X eigenstates \|±⟩ to \|0⟩, \|1⟩; S†H does the same for the Y eigenstates \|±i⟩) |
| `parity = sum(...) % 2` | the eigenvalue of a Pauli string on a bitstring is the product of the single-qubit eigenvalues (−1)ᵇ, i.e. (−1)^(number of 1s) on the non-identity positions — identity positions are skipped |
| `zip(bits, label)` | both strings are little-endian, so character k of the bitstring and character k of the label refer to the same qubit |
| the printed value < 2 | on `fake_apollo` each ⟨ZZ⟩ and ⟨XX⟩ comes out around 0.75–0.85 instead of 1: every CZ error, PRX error or readout error flips a parity and costs 2/shots in the average. Expect ≈ 1.5–1.7 in total; on `garnet` / `emerald` the number is a direct measure of the Bell-pair quality; `garnet:mock` returns random bitstrings, i.e. ≈ 0 |

</details>


## 🔬 In-depth analysis

1. **Pauli algebra.** With `Pauli` objects check that `X @ Y` is `iZ` (the matrix product XY = iZ) and that X and Z anticommute (`X.anticommutes(Z)`), then confirm the first identity numerically with `np.allclose` on the matrices.
2. **A Hamiltonian.** Build the two-qubit Heisenberg Hamiltonian H = XX + YY + ZZ as a `SparsePauliOp` and compute ⟨H⟩ exactly (`Statevector` and the Estimator) for the Bell state Φ⁺ and for the singlet Ψ⁻ = (\|01⟩ − \|10⟩)/√2 (prepared by `x(0); x(1); h(0); cx(0, 1)`). Which one is the ground state, and what is its energy?
3. **From counts.** Estimate ⟨H⟩ of the singlet with `pauli_expectation` (three measurement bases) and compare with the exact value.

Write your code in the empty cell below; a reference solution follows it. Try first, then compare.


In [ ]:
# 🔬 In-depth analysis — your code here



In [ ]:
# Reference solution
# (1) Pauli algebra with symbolic Pauli objects
X, Y, Z = Pauli('X'), Pauli('Y'), Pauli('Z')
XY = X @ Y                                                       # @ = matrix product (X.dot(Y)); the result is a Pauli with a phase: 'iZ'
print(f'(1) X @ Y = {XY} | matrices agree with i*Z: {np.allclose(XY.to_matrix(), 1j * Z.to_matrix())} | '
      f'X anticommutes with Z: {X.anticommutes(Z)} | X commutes with Z: {X.commutes(Z)}')

# (2) the Heisenberg Hamiltonian on the Bell state and on the singlet
H_heis = SparsePauliOp.from_list([('XX', 1.0), ('YY', 1.0), ('ZZ', 1.0)])
bell = QuantumCircuit(2); bell.h(0); bell.cx(0, 1)                       # |Φ+> = (|00> + |11>)/sqrt2
singlet = QuantumCircuit(2); singlet.x(0); singlet.x(1); singlet.h(0); singlet.cx(0, 1)   # |Ψ-> = (|01> - |10>)/sqrt2 up to a global sign
e_bell = Statevector(bell).expectation_value(H_heis).real                # exact <H>
e_singlet = Statevector(singlet).expectation_value(H_heis).real
res = estimator.run([(pm.run(bell), H_heis), (pm.run(singlet), H_heis)]).result()   # two pubs in one Estimator call
print(f'(2) <H> Phi+ = {e_bell:+.4f}, singlet = {e_singlet:+.4f}   (Estimator: {float(res[0].data.evs):+.4f}, {float(res[1].data.evs):+.4f})'
      f' -> the singlet is the ground state, energy -3 (the eigenvalues of H are +1, +1, +1, -3)')
display(draw(singlet)); print("↑ the singlet circuit: X X puts the register in |11>, H on qubit 0 and the CNOT entangle it into (|10> − |01>)/√2")

# (3) the singlet energy from counts: one measurement basis per Pauli string
e_counts = sum(c.real * pauli_expectation(singlet, p.to_label()) for p, c in zip(H_heis.paulis, H_heis.coeffs))
print(f'(3) <H> singlet from counts = {e_counts:+.4f}   (exact -3; every term is -1 for the singlet, so any deviation is noise)')


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `X @ Y` → `iZ` | Pauli objects carry a phase: XY = iZ, YZ = iX, ZX = iY (cyclic), and XY = −YX. `anticommutes` / `commutes` test the relation without building matrices — this is what makes Pauli-string arithmetic cheap for 50 qubits |
| `H_heis = XX + YY + ZZ` | the Heisenberg exchange interaction; on two qubits its eigenstates are the Bell states: the triplet (Φ⁺, Φ⁻, Ψ⁺) at energy +1 and the singlet Ψ⁻ at −3 |
| `estimator.run([pub1, pub2])` | several pubs in one call: `res[k].data.evs` is the value of pub k — the pattern of a VQE energy evaluation |
| `(3)` vs `(2)` | the same number from three measurement settings (X, Y, Z basis). On the ideal simulator every shot of the singlet has odd parity in all three bases, so the estimate is exactly −3; on a noisy backend each flipped parity pulls it towards 0 |

</details>


<details open><summary><b>📐 Working it out</b></summary>

On $|\Phi^+\rangle=(|00\rangle+|11\rangle)/\sqrt2$: $\langle XX\rangle=+1$ and $\langle ZZ\rangle=+1$ (box above), while
$Y\otimes Y\,|00\rangle = (i|1\rangle)(i|1\rangle) = -|11\rangle$ and $Y\otimes Y|11\rangle=(-i)^2|00\rangle=-|00\rangle$, so $\langle YY\rangle=-1$:

$$\langle H\rangle_{\Phi^+}=1-1+1=+1 .$$

On the singlet $|\Psi^-\rangle=(|01\rangle-|10\rangle)/\sqrt2$ the two bits always differ, so $\langle ZZ\rangle=-1$; $X\otimes X$ swaps the two terms and picks up the minus sign, $\langle XX\rangle=-1$; the same for $Y\otimes Y$:

$$\langle H\rangle_{\Psi^-}=-1-1-1=-3 ,$$

the ground-state energy printed in (2) and estimated in (3).

</details>


## 📖 References

1. **Operator API**: https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Operator
2. **Estimator Guide**: https://docs.quantum.ibm.com/guides/primitives-examples

---

**Course**: Quantum Computing Laboratory  
**License**: MIT  
**Version**: 2.2.0  
**Last Updated**: November 2025